# Olist Data Structure and Quality Check

## Day 40

### 목적

Olist 공개 데이터를 실제 분석에 사용하기 전에
각 테이블의 구조, 행 단위, Key와 데이터 품질을 확인한다.

### 확인 질문

1. 각 CSV의 한 행은 무엇을 의미하는가?
2. Primary Key는 실제로 중복되지 않는가?
3. customer_id와 customer_unique_id는 어떻게 다른가?
4. 주요 컬럼에 결측값이 존재하는가?
5. 주문 날짜의 실제 범위는 언제인가?
6. 어떤 주문 상태가 존재하는가?
7. 가격에 비정상 값이 존재하는가?
8. Foreign Key 연결에 누락이 존재하는가?
9. 실제 분석에 사용할 수 있는 기간과 지표는 무엇인가?

In [1]:
from pathlib import Path
import pandas as pd

In [4]:
project_root = Path.cwd().parent

raw_dir = (
    project_root
    / "data"
    / "raw"
    / "olist"
)

report_dir = (
    project_root
    / "reports"
)

report_dir.mkdir(
    parents = True,
    exist_ok = True
)

print(
    project_root
)

print(
    raw_dir
)

d:\Study\Projects\01_sales_analysis
d:\Study\Projects\01_sales_analysis\data\raw\olist


In [9]:
customers_path = (
    raw_dir
    / "olist_customers_dataset.csv"
)

orders_path = (
    raw_dir
    / "olist_orders_dataset.csv"
)

order_items_path = (
    raw_dir
    / "olist_order_items_dataset.csv"
)

products_path = (
    raw_dir
    / "olist_products_dataset.csv"
)

category_translation_path = (
    raw_dir
    / "product_category_name_translation.csv"
)

In [10]:
print(
    customers_path.exists()
)

print(
    orders_path.exists()
)

print(
    order_items_path.exists()
)

print(
    products_path.exists()
)

print(
    category_translation_path.exists()
)

True
True
True
True
True


In [11]:
customers_df = pd.read_csv(
    customers_path
)

orders_df = pd.read_csv(
    orders_path
)

order_items_df = pd.read_csv(
    order_items_path
)

products_df = pd.read_csv(
    products_path
)

category_translation_df = pd.read_csv(
    category_translation_path
)

In [ ]:
# 행렬 확인
print(
    "customers:",
    customers_df.shape
)

print(
    "orders:",
    orders_df.shape
)

print(
    "order_items:",
    order_items_df.shape
)

print(
    "products:",
    products_df.shape
)

print(
    "category_translation:",
    category_translation_df.shape
)

customers: (99441, 5)
orders: (99441, 8)
order_items: (112650, 7)
products: (32951, 9)
category_translation: (71, 2)


In [ ]:
# 컬럼 확인
print(
    customers_df.columns
)

print(
    orders_df.columns
)

print(
    order_items_df.columns
)

print(
    products_df.columns
)

print(
    category_translation_df.columns
)

Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='str')
Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')
Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='str')
Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='str')
Index(['product_category_name', 'product_category_name_english'], dtype='str')


In [ ]:
# Dtype 확인
print(
    customers_df.dtypes
)

print(
    orders_df.dtypes
)

print(
    order_items_df.dtypes
)

print(
    products_df.dtypes
)

customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object
product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_h

In [ ]:
# Customers Grain 
print(
    "customer_id:",
    customers_df["customer_id"].nunique()
)

print(
    "customer_unique_id:",
    customers_df["customer_unique_id"].nunique()
)

print(
    "rows:",
    len(customers_df)
)

customer_id: 99441
customer_unique_id: 96096
rows: 99441


In [ ]:
# Customer Key 중복
print(
    "customer_id duplicates:",
    customers_df["customer_id"]
    .duplicated()
    .sum()
)

print(
    "customer_unique_id duplicates:",
    customers_df["customer_unique_id"]
    .duplicated()
    .sum()
)

customer_id duplicates: 0
customer_unique_id duplicates: 3345


In [ ]:
# Orders Primary Key 주문 1건당 1행
print(
    "order rows:",
    len(orders_df)
)

print(
    "unique order_id:",
    orders_df["order_id"].nunique()
)

print(
    "duplicate order_id:",
    orders_df["order_id"]
    .duplicated()
    .sum()
)

print(
    orders_df["order_id"].is_unique
)

order rows: 99441
unique order_id: 99441
duplicate order_id: 0
True


In [ ]:
# Order Items Grain 주문 안의 상품 항목 1행
print(
    "rows:",
    len(order_items_df)
)

print(
    "unique order_id:",
    order_items_df["order_id"].nunique()
)

print(
    "duplicate order_id:",
    order_items_df["order_id"]
    .duplicated()
    .sum()
)

print(
    "duplicate order item key:",
    order_items_df[
        [
            "order_id",
            "order_item_id"
        ]
    ]
    .duplicated()
    .sum()
)

rows: 112650
unique order_id: 98666
duplicate order_id: 13984
duplicate order item key: 0


In [ ]:
# Products Key 상품 1개당 1행
print(
    "product rows:",
    len(products_df)
)

print(
    "unique product_id:",
    products_df["product_id"].nunique()
)

print(
    "duplicate product_id:",
    products_df["product_id"]
    .duplicated()
    .sum()
)

product rows: 32951
unique product_id: 32951
duplicate product_id: 0


In [ ]:
# 전체 행 중복 확인
print(
    "customers full duplicates:",
    customers_df
    .duplicated()
    .sum()
)

print(
    "orders full duplicates:",
    orders_df
    .duplicated()
    .sum()
)

print(
    "order_items full duplicates:",
    order_items_df
    .duplicated()
    .sum()
)

print(
    "products full duplicates:",
    products_df
    .duplicated()
    .sum()
)

customers full duplicates: 0
orders full duplicates: 0
order_items full duplicates: 0
products full duplicates: 0


In [ ]:
# 결측값
print(
    customers_df
    .isna()
    .sum()
)

print(
    orders_df
    .isna()
    .sum()
)

print(
    order_items_df
    .isna()
    .sum()
)

print(
    products_df
    .isna()
    .sum()
)

print(
    category_translation_df
    .isna()
    .sum()
)

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64
product_categ

In [ ]:
# 날짜 변환
orders_df["order_purchase_timestamp"] = pd.to_datetime(
    orders_df["order_purchase_timestamp"],
    errors="coerce"
)

np.int64(0)

In [ ]:
# 날짜 범위
print(
    "date conversion missing:",
    orders_df["order_purchase_timestamp"]
    .isna()
    .sum()
)

print(
    "min date:",
    orders_df["order_purchase_timestamp"]
    .min()
)

print(
    "max date:",
    orders_df["order_purchase_timestamp"]
    .max()
)

date conversion missing: 0
min date: 2016-09-04 21:15:19
max date: 2018-10-17 17:30:18


In [27]:
orders_df["order_month"] = (
    orders_df["order_purchase_timestamp"]
    .dt.to_period("M")
)

In [ ]:
# 월별 데이터량 확인
monthly_order_count = (
    orders_df["order_month"]
    .value_counts()
    .sort_index()
)

print(
    monthly_order_count
)

order_month
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, Name: count, dtype: int64


In [ ]:
# 주문 상태 확인
order_status_count = (
    orders_df["order_status"]
    .value_counts(
        dropna=False
    )
)

print(
    order_status_count
)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [ ]:
# 가격품질 확인
print(
    order_items_df["price"]
    .describe()
)

print(
    "negative price:",
    (
        order_items_df["price"] < 0
    )
    .sum()
)

print(
    "zero price:",
    (
        order_items_df["price"] == 0
    )
    .sum()
)

print(
    "negative freight:",
    (
        order_items_df["freight_value"] < 0
    )
    .sum()
)

count    112650.000000
mean        120.653739
std         183.633928
min           0.850000
25%          39.900000
50%          74.990000
75%         134.900000
max        6735.000000
Name: price, dtype: float64
negative price: 0
zero price: 0
negative freight: 0


In [33]:
# Foreign Key 검증

### orders -> customers
orders_customer_unmatched = (
    ~orders_df["customer_id"]
    .isin(
        customers_df["customer_id"]
    )
)

print(
    "orders -> customers unmatched:",
    orders_customer_unmatched.sum()
)

### order_items -> orders
items_order_unmatched = (
    ~order_items_df["order_id"]
    .isin(
        orders_df["order_id"]
    )
)

print(
    "order_items -> orders unmatched:",
    items_order_unmatched.sum()
)

### order_items -> products
items_product_unmatched = (
    ~order_items_df["product_id"]
    .isin(
        products_df["product_id"]
    )
)

print(
    "order_items -> products unmatched:",
    items_product_unmatched.sum()
)

### products -> category translation
category_unmatched = (
    products_df["product_category_name"]
    .notna()
    &
    ~products_df["product_category_name"]
    .isin(
        category_translation_df[
            "product_category_name"
        ]
    )
)

print(
    "category translation unmatched:",
    category_unmatched.sum()
)

orders -> customers unmatched: 0
order_items -> orders unmatched: 0
order_items -> products unmatched: 0
category translation unmatched: 13


In [34]:
# Join 위험 확인
print(
    "orders rows:",
    len(orders_df)
)

print(
    "order_items rows:",
    len(order_items_df)
)

print(
    "order_items unique orders:",
    order_items_df["order_id"].nunique()
)

multi_item_orders = (
    order_items_df["order_id"]
    .value_counts()
    > 1
)

print(
    "orders with multiple item rows:",
    multi_item_orders.sum()
)

orders rows: 99441
order_items rows: 112650
order_items unique orders: 98666
orders with multiple item rows: 9803


In [35]:
# 품질 요약표
quality_summary_df = pd.DataFrame(
    {
        "table_name": [
            "customers",
            "orders",
            "order_items",
            "products",
            "category_translation"
        ],
        "row_count": [
            len(customers_df),
            len(orders_df),
            len(order_items_df),
            len(products_df),
            len(category_translation_df)
        ],
        "column_count": [
            len(customers_df.columns),
            len(orders_df.columns),
            len(order_items_df.columns),
            len(products_df.columns),
            len(category_translation_df.columns)
        ],
        "full_duplicate_count": [
            customers_df.duplicated().sum(),
            orders_df.duplicated().sum(),
            order_items_df.duplicated().sum(),
            products_df.duplicated().sum(),
            category_translation_df.duplicated().sum()
        ]
    }
)

print(
    quality_summary_df
)

             table_name  row_count  column_count  full_duplicate_count
0             customers      99441             5                     0
1                orders      99441             9                     0
2           order_items     112650             7                     0
3              products      32951             9                     0
4  category_translation         71             2                     0


In [36]:
# 저장
quality_summary_path = (
    report_dir
    / "data_quality_summary.csv"
)

quality_summary_df.to_csv(
    quality_summary_path,
    index=False
)

print(
    quality_summary_path
)

d:\Study\Projects\01_sales_analysis\reports\data_quality_summary.csv


## Day 40 분석 결과

### 데이터 구조

- customers: (99441, 5)
- orders: (99441, 8)
- order_items: (112650, 7)
- products: (32951, 9)
- category_translation: (71, 2)

### Key 검증

- customer_id 중복: 0
- customer_unique_id 고유 고객 수: 96096
- order_id 중복: 13984
- order_id + order_item_id 중복: 0
- product_id 중복: 0

### Foreign Key 검증

- orders → customers 미매칭: 0
- order_items → orders 미매칭: 0
- order_items → products 미매칭: 0
- products → category translation 미매칭: 13

### 날짜

- 최초 주문일: 2016-09-04
- 마지막 주문일: 2018-10-17
- 날짜 변환 실패: 0
- 실제 분석 후보 기간: 2017년 1월 ~ 2017년 12월 완전한 최근 12개월

### 주문 상태

존재하는 주문 상태: 

- delivered      96478
- shipped         1107
- canceled         625
- unavailable      609
- invoiced         314
- processing       301
- created            5
- approved           2

### 가격

- 최소 가격: 0.85
- 최대 가격: 6735.00
- 음수 가격: 0
- 0 가격: 0

### 핵심 해석

Olist 데이터는 주문, 주문상품, 고객과 상품 테이블을
관계형 구조로 연결할 수 있는 데이터이다.

orders와 order_items는 Grain이 다르므로
JOIN 이후 행 수 증가를 정상적인 주문상품 확장과
비정상적인 중복으로 구분해야 한다.

customer_id와 customer_unique_id의 역할이 다르므로
주문 연결에는 customer_id를 사용하고,
실제 고객 단위 재구매 분석에는 customer_unique_id를 사용해야 한다.

실제 분석에 사용하기 전에
주문 상태와 분석 기간을 추가로 정의해야 한다.